# Protocol Spec Assist — Google Colab Runner

**Run the full pipeline on a free/Pro Colab GPU.**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU** (free) or **V100/A100** (Pro)
2. Have your OCR'd protocol PDF ready to upload

Then run each cell top to bottom.

---
## Step 1: Check GPU
Make sure you got a GPU assigned.

In [ ]:
!nvidia-smi
# You should see a T4 (16 GB), V100 (16 GB), or A100 (40/80 GB)
# If you see an error, go to Runtime → Change runtime type → GPU

---
## Step 2: Clone the project

In [ ]:
# Clone the repo (change URL to your fork/repo if different)
!git clone https://github.com/salzburry/qcagent.git
%cd qcagent

---
## Step 3: Install dependencies
This takes 3-5 minutes.

In [ ]:
# Install the project + all dependencies
!pip install -e ".[dev]" --quiet

# Install vLLM (the LLM server)
!pip install vllm --quiet

print("\n✅ Dependencies installed.")

---
## Step 4: Upload your protocol PDF
Run this cell — a file picker will appear. Select your OCR'd PDF.

In [ ]:
import os
from google.colab import files

# Create the protocols directory
os.makedirs("data/protocols", exist_ok=True)

# Upload your PDF
print("Select your OCR'd protocol PDF...")
uploaded = files.upload()

# Move uploaded file to the protocols folder
for filename in uploaded:
    os.rename(filename, f"data/protocols/{filename}")
    print(f"\n✅ Uploaded: data/protocols/{filename}")

# Store the filename for later use
PDF_FILENAME = list(uploaded.keys())[0]
PDF_PATH = f"data/protocols/{PDF_FILENAME}"
PROTOCOL_ID = os.path.splitext(PDF_FILENAME)[0]

print(f"\nProtocol ID: {PROTOCOL_ID}")
print(f"PDF path:    {PDF_PATH}")

---
## Step 5: Start the LLM server (vLLM)
This starts vLLM in the background. Takes 2-3 minutes to load the model.

**On a free T4 (16 GB VRAM):** We use reduced context length to fit in memory.  
**On Pro V100/A100:** Full context length works fine.

In [ ]:
import subprocess, time

# Check GPU memory to pick settings
gpu_info = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits
gpu_mem_mb = int(gpu_info[0].strip())
print(f"GPU memory: {gpu_mem_mb} MB")

if gpu_mem_mb >= 40000:
    # A100 — full context, plenty of room
    MAX_MODEL_LEN = 32768
    print("A100 detected — using full context length.")
elif gpu_mem_mb >= 20000:
    # V100 — moderate context
    MAX_MODEL_LEN = 24576
    print("V100 detected — using 24K context length.")
else:
    # T4 (16 GB) — tight fit, reduce context
    MAX_MODEL_LEN = 16384
    print("T4 detected — using 16K context length to fit in 16 GB VRAM.")

# Start vLLM in the background
vllm_proc = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen3-8B",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--enable-prefix-caching",
        "--dtype", "auto",
        "--gpu-memory-utilization", "0.95",
    ],
    stdout=open("vllm_stdout.log", "w"),
    stderr=open("vllm_stderr.log", "w"),
)

print(f"\nvLLM starting (PID: {vllm_proc.pid})...")
print("Waiting for model to load (2-3 minutes)...")

In [ ]:
# Wait for vLLM to be ready
import urllib.request, time

ready = False
for i in range(60):  # wait up to 5 minutes
    try:
        resp = urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        if resp.status == 200:
            ready = True
            break
    except Exception:
        pass
    if i % 6 == 0:
        print(f"  Waiting... ({i * 5}s elapsed)")
    time.sleep(5)

if ready:
    print("\n✅ vLLM is ready!")
else:
    print("\n❌ vLLM failed to start. Check logs:")
    !tail -30 vllm_stderr.log

---
## Step 6: Set environment variables

In [ ]:
import os

os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["ADJUDICATOR_BASE_URL"] = "http://localhost:8000/v1"  # same model for both
os.environ["VLLM_API_KEY"] = "local"
os.environ["DEFAULT_MODEL"] = "Qwen/Qwen3-8B"
os.environ["ADJUDICATOR_MODEL"] = "Qwen/Qwen3-8B"

print("✅ Environment configured.")

---
## Step 7: Run the pipeline

**Change the `--ta` flag** to match your protocol's therapeutic area:  
`oncology` | `cardiovascular` | `respiratory` | `immunology` | `vaccines`  

Or remove `--ta` entirely if none apply.

This runs all 12 concept finders. Takes **10-30 minutes** depending on GPU and protocol length.

In [ ]:
# === CONFIGURE THESE ===
TA = "oncology"  # Change to your therapeutic area, or set to None
DATA_SOURCE = None  # Optional: "cota" | "flatiron" | "optum_cdm" | "optum_ehr" | "marketscan" | "inalon" | "quest"
# =======================

cmd = f"python -m protocol_spec_assist.workflows.protocol_run '{PDF_PATH}' --protocol-id '{PROTOCOL_ID}'"

if TA:
    cmd += f" --ta {TA}"
if DATA_SOURCE:
    cmd += f" --data-source {DATA_SOURCE}"

print(f"Running: {cmd}\n")
!{cmd}

---
## Step 8: Check outputs

In [ ]:
import os

output_dir = "data/outputs"
print("Generated files:")
print("=" * 50)
for f in sorted(os.listdir(output_dir)):
    if PROTOCOL_ID in f:
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f}  ({size:,} bytes)")

---
## Step 9: Preview the HTML spec

In [ ]:
from IPython.display import HTML, display

html_path = f"data/outputs/{PROTOCOL_ID}_spec.html"
if os.path.exists(html_path):
    with open(html_path, "r") as f:
        display(HTML(f.read()))
else:
    print("HTML spec not found — check the pipeline output above for errors.")

---
## Step 10: Download all output files to your laptop

In [ ]:
from google.colab import files
import os

output_dir = "data/outputs"
for f in sorted(os.listdir(output_dir)):
    if PROTOCOL_ID in f:
        path = os.path.join(output_dir, f)
        print(f"Downloading: {f}")
        files.download(path)

print("\nDone! Check your browser's Downloads folder.")

---
## Troubleshooting

**vLLM out of memory on T4?** Run this cell to check the log, then try the fix below.

In [ ]:
# Check vLLM logs for errors
!tail -50 vllm_stderr.log

**If vLLM crashes on T4**, kill it and restart with 4-bit quantization:

In [ ]:
# Only run this if the T4 ran out of memory above
# Kill existing vLLM
!pkill -f "vllm serve" 2>/dev/null
import time; time.sleep(3)

# Install quantized model support
!pip install autoawq --quiet

import subprocess

# Restart with quantized model (fits easily in 16 GB)
vllm_proc = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen3-8B-AWQ",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--max-model-len", "16384",
        "--enable-prefix-caching",
        "--dtype", "auto",
        "--quantization", "awq",
    ],
    stdout=open("vllm_stdout.log", "w"),
    stderr=open("vllm_stderr.log", "w"),
)

print("Restarting with quantized model (AWQ 4-bit)...")
print("Wait 2-3 minutes, then re-run the 'Wait for vLLM' cell above.")
print("\nIMPORTANT: Also update the model name:")
print('  os.environ["DEFAULT_MODEL"] = "Qwen/Qwen3-8B-AWQ"')
print('  os.environ["ADJUDICATOR_MODEL"] = "Qwen/Qwen3-8B-AWQ"')